In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

with open(PROJECT_ROOT / "DataPipeline" / "config.yml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

data_path = PROJECT_ROOT / cfg["paths"]["raw_dir"]

# Endpoint/credenciais do MinIO: variáveis de ambiente têm precedência (dentro
# do Docker o compose já as define); fora, caem nos padrões do stack local.
os.environ.setdefault("MINIO_ENDPOINT", "http://localhost:9000")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "minioadmin")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "minioadmin123")


def carregar_raw(nome_arquivo):
    """CSV bruto: usa a cópia local (Dados/raw) se existir; senão lê do bucket
    `raw` do MinIO (populado pelo seeder)."""
    local = data_path / nome_arquivo
    if local.exists():
        return pd.read_csv(local)
    from storage import get_storage
    with get_storage(cfg).open("raw", nome_arquivo, "rb") as fh:
        return pd.read_csv(fh)

In [2]:
def optimize_dtypes(df):
    """
    Otimiza os tipos de dados para reduzir o consumo de memória.

    Operações realizadas:
        - float64 -> float32
        - int64 -> menor tipo inteiro possível (downcast)

    """

    # Converte colunas float64 para float32
    colunas_float = df.select_dtypes(include="float64").columns

    if len(colunas_float) > 0:
        df[colunas_float] = df[colunas_float].apply(
            pd.to_numeric,
            downcast="float"
        )

    # Converte colunas inteiras para o menor tipo possível
    colunas_int = df.select_dtypes(include="int64").columns

    for coluna in colunas_int:
        df[coluna] = pd.to_numeric(
            df[coluna],
            downcast="integer"
        )

    return df

In [3]:
def clean_dataframe(df):
    """
    Executa operações genéricas de limpeza aplicáveis a qualquer DataFrame.

    Operações realizadas:
        - Remove linhas duplicadas
        - Remove colunas duplicadas
        - Remove espaços em branco no início e fim de colunas de texto
    """

    # Remove linhas completamente duplicadas
    df = df.drop_duplicates()

    # Remove espaços em branco das colunas de texto
    colunas_texto = df.select_dtypes(include="object").columns

    for coluna in colunas_texto:
        df[coluna] = df[coluna].str.strip()

    # Remove colunas duplicadas, caso existam
    df = df.loc[:, ~df.columns.duplicated()]

    return df

In [4]:
raw_files = cfg["data"]["raw_files"]

df_train = carregar_raw(raw_files["application"])
df_bureau = carregar_raw(raw_files["bureau"])
df_prevapp = carregar_raw(raw_files["previous_application"])

In [5]:
df_train = (
    optimize_dtypes(df_train)
    .pipe(clean_dataframe)
)

In [6]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 122 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float32(64), float64(1), int16(2), int32(2), int8(37), object(16)
memory usage: 129.3+ MB


In [7]:
df_train.head(30)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.000,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.000,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.000,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.000,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.000,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
5,100008,0,Cash loans,M,N,Y,0,99000.000,490495.5,27517.5,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0
6,100009,0,Cash loans,F,Y,Y,1,171000.000,1560726.0,41301.0,...,0,0,0,0,0.0,0.0,0.0,1.0,1.0,2.0
7,100010,0,Cash loans,M,Y,Y,0,360000.000,1530000.0,42075.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
8,100011,0,Cash loans,F,N,Y,0,112500.000,1019610.0,33826.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
9,100012,0,Revolving loans,M,N,Y,0,135000.000,405000.0,20250.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df_train.describe()

,SK_ID_CURR,TARGET,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
count,307511.000000,307511.000000,307511.000000,3.075110e+05,3.075110e+05,307499.000000,3.072330e+05,307511.000000,307511.000000,307511.000000,...,307511.000000,307511.000000,307511.000000,307511.000000,265992.000000,265992.000000,265992.000000,265992.000000,265992.000000,265992.000000
mean,278180.518577,0.080729,0.417052,1.687979e+05,5.990259e+05,27108.572266,5.383961e+05,0.020868,-16036.995067,63815.045904,...,0.008130,0.000595,0.000507,0.000335,0.006402,0.007000,0.034362,0.267395,0.265474,1.899974
std,102790.175348,0.272419,0.722121,2.371231e+05,4.024908e+05,14493.737305,3.694465e+05,0.013831,4363.988632,141275.766519,...,0.089798,0.024387,0.022518,0.018299,0.083849,0.110757,0.204685,0.916002,0.794056,1.869295
min,100002.000000,0.000000,0.000000,2.565000e+04,4.500000e+04,1615.500000,4.050000e+04,0.000290,-25229.000000,-17912.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,189145.500000,0.000000,0.000000,1.125000e+05,2.700000e+05,16524.000000,2.385000e+05,0.010006,-19682.000000,-2760.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,278202.000000,0.000000,0.000000,1.471500e+05,5.135310e+05,24903.000000,4.500000e+05,0.018850,-15750.000000,-1213.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,367142.500000,0.000000,1.000000,2.025000e+05,8.086500e+05,34596.000000,6.795000e+05,0.028663,-12413.000000,-289.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000
max,456255.000000,1.000000,19.000000,1.170000e+08,4.050000e+06,258025.500000,4.050000e+06,0.072508,-7489.000000,365243.000000,...,1.000000,1.000000,1.000000,1.000000,4.000000,9.000000,8.000000,27.000000,261.000000,25.000000


In [9]:
df_train["CODE_GENDER"].value_counts()

CODE_GENDER
F      202448
M      105059
XNA         4
Name: count, dtype: int64

In [10]:
df_train["FLAG_OWN_CAR"].value_counts()

FLAG_OWN_CAR
N    202924
Y    104587
Name: count, dtype: int64

In [11]:
df_train["NAME_FAMILY_STATUS"].value_counts()

NAME_FAMILY_STATUS
Married                 196432
Single / not married     45444
Civil marriage           29775
Separated                19770
Widow                    16088
Unknown                      2
Name: count, dtype: int64

In [12]:
df_train.isna().mean().sort_values(ascending=False).head(20)

COMMONAREA_MEDI             0.698723
COMMONAREA_AVG              0.698723
COMMONAREA_MODE             0.698723
NONLIVINGAPARTMENTS_MODE    0.694330
NONLIVINGAPARTMENTS_AVG     0.694330
NONLIVINGAPARTMENTS_MEDI    0.694330
FONDKAPREMONT_MODE          0.683862
LIVINGAPARTMENTS_MODE       0.683550
LIVINGAPARTMENTS_AVG        0.683550
LIVINGAPARTMENTS_MEDI       0.683550
FLOORSMIN_AVG               0.678486
FLOORSMIN_MODE              0.678486
FLOORSMIN_MEDI              0.678486
YEARS_BUILD_MEDI            0.664978
YEARS_BUILD_MODE            0.664978
YEARS_BUILD_AVG             0.664978
OWN_CAR_AGE                 0.659908
LANDAREA_MEDI               0.593767
LANDAREA_MODE               0.593767
LANDAREA_AVG                0.593767
dtype: float64

In [13]:
df_train["OBS_30_CNT_SOCIAL_CIRCLE"].isna().sum()

1021

In [14]:
missing = pd.DataFrame({
    "Missing": df_train.isna().sum(),
    "Percent": (df_train.isna().mean() * 100).round(2)
})

missing = missing[missing["Missing"] > 0].sort_values(
    by="Percent",
    ascending=False
)

print(missing)

                          Missing  Percent
COMMONAREA_MEDI            214865    69.87
COMMONAREA_AVG             214865    69.87
COMMONAREA_MODE            214865    69.87
NONLIVINGAPARTMENTS_MEDI   213514    69.43
NONLIVINGAPARTMENTS_MODE   213514    69.43
...                           ...      ...
EXT_SOURCE_2                  660     0.21
AMT_GOODS_PRICE               278     0.09
DAYS_LAST_PHONE_CHANGE          1     0.00
CNT_FAM_MEMBERS                 2     0.00
AMT_ANNUITY                    12     0.00

[67 rows x 2 columns]


In [15]:
missing[missing["Percent"] > 50]

,Missing,Percent
COMMONAREA_MEDI,214865,69.87
COMMONAREA_AVG,214865,69.87
COMMONAREA_MODE,214865,69.87
NONLIVINGAPARTMENTS_MEDI,213514,69.43
NONLIVINGAPARTMENTS_MODE,213514,69.43
NONLIVINGAPARTMENTS_AVG,213514,69.43
FONDKAPREMONT_MODE,210295,68.39
LIVINGAPARTMENTS_MODE,210199,68.35
LIVINGAPARTMENTS_MEDI,210199,68.35
LIVINGAPARTMENTS_AVG,210199,68.35


In [16]:
df_bureau.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1716428 entries, 0 to 1716427
Data columns (total 17 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_CURR              int64  
 1   SK_ID_BUREAU            int64  
 2   CREDIT_ACTIVE           object 
 3   CREDIT_CURRENCY         object 
 4   DAYS_CREDIT             int64  
 5   CREDIT_DAY_OVERDUE      int64  
 6   DAYS_CREDIT_ENDDATE     float64
 7   DAYS_ENDDATE_FACT       float64
 8   AMT_CREDIT_MAX_OVERDUE  float64
 9   CNT_CREDIT_PROLONG      int64  
 10  AMT_CREDIT_SUM          float64
 11  AMT_CREDIT_SUM_DEBT     float64
 12  AMT_CREDIT_SUM_LIMIT    float64
 13  AMT_CREDIT_SUM_OVERDUE  float64
 14  CREDIT_TYPE             object 
 15  DAYS_CREDIT_UPDATE      int64  
 16  AMT_ANNUITY             float64
dtypes: float64(8), int64(6), object(3)
memory usage: 222.6+ MB


In [17]:
df_bureau.head(15)

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.00,0.00,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.00,171342.00,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.50,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.00,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.00,NaN,NaN,0.0,Consumer credit,-21,NaN
5,215354,5714467,Active,currency 1,-273,0,27460.0,NaN,0.0,0,180000.00,71017.38,108982.62,0.0,Credit card,-31,NaN
6,215354,5714468,Active,currency 1,-43,0,79.0,NaN,0.0,0,42103.80,42103.80,0.00,0.0,Consumer credit,-22,NaN
7,162297,5714469,Closed,currency 1,-1896,0,-1684.0,-1710.0,14985.0,0,76878.45,0.00,0.00,0.0,Consumer credit,-1710,NaN
8,162297,5714470,Closed,currency 1,-1146,0,-811.0,-840.0,0.0,0,103007.70,0.00,0.00,0.0,Consumer credit,-840,NaN
9,162297,5714471,Active,currency 1,-1146,0,-484.0,NaN,0.0,0,4500.00,0.00,0.00,0.0,Credit card,-690,NaN


In [18]:
df_bureau["SK_ID_BUREAU"].is_unique

True

In [19]:
assert df_bureau["SK_ID_CURR"].notna().all()
assert df_bureau["SK_ID_BUREAU"].notna().all()

In [20]:
df_bureau["AMT_CREDIT_SUM_OVERDUE"] < 0

0          False
1          False
2          False
3          False
4          False
           ...  
1716423    False
1716424    False
1716425    False
1716426    False
1716427    False
Name: AMT_CREDIT_SUM_OVERDUE, Length: 1716428, dtype: bool

In [21]:
future = df_bureau["DAYS_CREDIT"] > 0
future.info()

<class 'pandas.core.series.Series'>
RangeIndex: 1716428 entries, 0 to 1716427
Series name: DAYS_CREDIT
Non-Null Count    Dtype
--------------    -----
1716428 non-null  bool 
dtypes: bool(1)
memory usage: 1.6 MB


In [22]:
def sanitize_bureau(df: pd.DataFrame) -> pd.DataFrame:
    """
    Sanitiza a tabela bureau.

    Operações realizadas:
        - Remove registros sem identificadores
        - Substitui valores monetários negativos por NA
        - Substitui contagens negativas de prorrogação por NA
        - Substitui datas futuras (dias positivos) por NA
    """

    # Remove registros sem identificadores
    df = df.dropna(subset=["SK_ID_CURR", "SK_ID_BUREAU"])

    # Valores monetários não podem ser negativos
    monetary_cols = [
        "AMT_CREDIT_SUM",
        "AMT_CREDIT_SUM_DEBT",
        "AMT_CREDIT_SUM_LIMIT",
        "AMT_CREDIT_SUM_OVERDUE",
    ]

    for col in monetary_cols:
        if col in df.columns:
            df.loc[df[col] < 0, col] = pd.NA

    # Quantidade de prorrogações não pode ser negativa
    if "CNT_CREDIT_PROLONG" in df.columns:
        df.loc[df["CNT_CREDIT_PROLONG"] < 0, "CNT_CREDIT_PROLONG"] = pd.NA

    # Datas em bureau representam dias relativos ao momento da aplicação.
    # Valores positivos indicariam eventos futuros.
    day_cols = [
        "DAYS_CREDIT",
        "DAYS_CREDIT_ENDDATE",
        "DAYS_ENDDATE_FACT",
        "DAYS_CREDIT_UPDATE",
    ]

    for col in day_cols:
        if col in df.columns:
            df.loc[df[col] > 0, col] = pd.NA

    return df

In [23]:
df_bureau2 = df_bureau.copy()

In [24]:
df_bureau2.equals(df_bureau)

True

In [25]:
sanitize_bureau(df_bureau2)

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497.0,0,-153.0,-153.0,NaN,0.0,91323.00,0.0,NaN,0.0,Consumer credit,-131.0,NaN
1,215354,5714463,Active,currency 1,-208.0,0,NaN,NaN,NaN,0.0,225000.00,171342.0,NaN,0.0,Credit card,-20.0,NaN
2,215354,5714464,Active,currency 1,-203.0,0,NaN,NaN,NaN,0.0,464323.50,NaN,NaN,0.0,Consumer credit,-16.0,NaN
3,215354,5714465,Active,currency 1,-203.0,0,NaN,NaN,NaN,0.0,90000.00,NaN,NaN,0.0,Credit card,-16.0,NaN
4,215354,5714466,Active,currency 1,-629.0,0,NaN,NaN,77674.5,0.0,2700000.00,NaN,NaN,0.0,Consumer credit,-21.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1716423,259355,5057750,Active,currency 1,-44.0,0,-30.0,NaN,0.0,0.0,11250.00,11250.0,0.0,0.0,Microloan,-19.0,NaN
1716424,100044,5057754,Closed,currency 1,-2648.0,0,-2433.0,-2493.0,5476.5,0.0,38130.84,0.0,0.0,0.0,Consumer credit,-2493.0,NaN
1716425,100044,5057762,Closed,currency 1,-1809.0,0,-1628.0,-970.0,NaN,0.0,15570.00,NaN,NaN,0.0,Consumer credit,-967.0,NaN
1716426,246829,5057770,Closed,currency 1,-1878.0,0,-1513.0,-1513.0,NaN,0.0,36000.00,0.0,0.0,0.0,Consumer credit,-1508.0,NaN


In [26]:
df_bureau.equals(df_bureau2)

True

In [27]:
df_prevapp.head()

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
def sanitize_previous_application(df: pd.DataFrame) -> pd.DataFrame:
    """
    Sanitiza a tabela previous_application.

    Operações realizadas:
        - Substitui o valor sentinela (365243) por NA nas colunas de dias
        - Substitui valores monetários negativos por NA
        - Substitui quantidade negativa de parcelas por NA
    """

    # Valor sentinela 365243 utilizado para representar ausência de informação
    day_cols = [
        "DAYS_FIRST_DRAWING",
        "DAYS_FIRST_DUE",
        "DAYS_LAST_DUE_1ST_VERSION",
        "DAYS_LAST_DUE",
        "DAYS_TERMINATION",
    ]

    for col in day_cols:
        if col in df.columns:
            df[col] = df[col].replace(365243, pd.NA)

    # Valores monetários não podem ser negativos
    monetary_cols = [
        "AMT_ANNUITY",
        "AMT_APPLICATION",
        "AMT_CREDIT",
        "AMT_DOWN_PAYMENT",
        "AMT_GOODS_PRICE",
    ]

    for col in monetary_cols:
        if col in df.columns:
            df.loc[df[col] < 0, col] = pd.NA

    # Número de parcelas não pode ser negativo
    if "CNT_PAYMENT" in df.columns:
        df.loc[df["CNT_PAYMENT"] < 0, "CNT_PAYMENT"] = pd.NA

    return df


In [29]:
day_cols = [
    "DAYS_FIRST_DRAWING",
    "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION",
    "DAYS_LAST_DUE",
    "DAYS_TERMINATION",
]

for col in day_cols:
    if col in df_prevapp.columns:
        count = (df_prevapp[col] == 365243).sum()
        print(f"{col}: {count:,}")

DAYS_FIRST_DRAWING: 934,444
DAYS_FIRST_DUE: 40,645
DAYS_LAST_DUE_1ST_VERSION: 93,864
DAYS_LAST_DUE: 211,221
DAYS_TERMINATION: 225,913


In [30]:
monetary_cols = [
    "AMT_ANNUITY",
    "AMT_APPLICATION",
    "AMT_CREDIT",
    "AMT_DOWN_PAYMENT",
    "AMT_GOODS_PRICE",
]

for col in monetary_cols:
    if col in df_prevapp.columns:
        count = (df_prevapp[col] < 0).sum()
        print(f"{col}: {count:,}")

AMT_ANNUITY: 0
AMT_APPLICATION: 0
AMT_CREDIT: 0
AMT_DOWN_PAYMENT: 2
AMT_GOODS_PRICE: 0


In [31]:
(df_prevapp["CNT_PAYMENT"] < 0).describe()

count     1670214
unique          1
top         False
freq      1670214
Name: CNT_PAYMENT, dtype: object

In [32]:
df_prevapp.loc[
    df_prevapp["AMT_CREDIT"] < 0,
    ["SK_ID_PREV", "AMT_CREDIT"]
]

,SK_ID_PREV,AMT_CREDIT


In [33]:
sanitized = sanitize_previous_application(df_prevapp.copy())

sanitized.isna().sum() - df_prevapp.isna().sum()

SK_ID_PREV                          0
SK_ID_CURR                          0
NAME_CONTRACT_TYPE                  0
AMT_ANNUITY                         0
AMT_APPLICATION                     0
AMT_CREDIT                          0
AMT_DOWN_PAYMENT                    2
AMT_GOODS_PRICE                     0
WEEKDAY_APPR_PROCESS_START          0
HOUR_APPR_PROCESS_START             0
FLAG_LAST_APPL_PER_CONTRACT         0
NFLAG_LAST_APPL_IN_DAY              0
RATE_DOWN_PAYMENT                   0
RATE_INTEREST_PRIMARY               0
RATE_INTEREST_PRIVILEGED            0
NAME_CASH_LOAN_PURPOSE              0
NAME_CONTRACT_STATUS                0
DAYS_DECISION                       0
NAME_PAYMENT_TYPE                   0
CODE_REJECT_REASON                  0
NAME_TYPE_SUITE                     0
NAME_CLIENT_TYPE                    0
NAME_GOODS_CATEGORY                 0
NAME_PORTFOLIO                      0
NAME_PRODUCT_TYPE                   0
CHANNEL_TYPE                        0
SELLERPLACE_

In [34]:
df_prevapp["DAYS_LAST_DUE"].describe()

count    997149.000000
mean      76582.403064
std      149647.415123
min       -2889.000000
25%       -1314.000000
50%        -537.000000
75%         -74.000000
max      365243.000000
Name: DAYS_LAST_DUE, dtype: float64